In [1]:
import random
import math
import hashlib
import struct
from typing import Callable, Any

In [2]:
class IntegerHash:
    def __init__(self):
        self._seed = random.randint(0, 2**32 - 1)

    def __call__(self, item: Any) -> int:
        text = str(item) + str(self._seed)
        hash_bytes = hashlib.md5(text.encode('utf-8')).digest()
        return struct.unpack('>I', hash_bytes[:4])[0]

In [3]:
class HLLSketch:
    def __init__(self, b: int, hashf: Callable[[Any], int]):
        self._b = b
        self._m = 1 << b
        self._hash = hashf
        self._registers = [0] * self._m

        self._alpha_hl = 0.39701
        self._alpha_hll = HLLSketch.get_alpha_hll(self._m)

    def add(self, item: Any):
        h = self._hash(item)
        j = h & (self._m - 1)
        w = h >> self._b
        self._registers[j] = max(self._registers[j], self._rho(w))

    def estimate_hyperlog(self) -> float:
        """
        E = alpha * m * 2^mean(register values) | Arithmetic Mean
        """
        arithmetic_mean = sum(self._registers) / self._m
        return self._alpha_hl * self._m * (2 ** arithmetic_mean)

    def estimate_hyperloglog(self) -> float:
        """
        E = alpha * m^2 * (sum(2^-r))^-1 | Harmonic Mean
        """
        sum_inverse_pow2 = sum(2 ** (-r) for r in self._registers)
        if sum_inverse_pow2 == 0:
            return 0.0

        return self._alpha_hll * (self._m ** 2) / sum_inverse_pow2

    def n_empty_registers(self) -> int:
        return self._registers.count(0)

    def _rho(self, w: int) -> int:
        """
        Calculates the position of the first true bit.
        """
        if w == 0:
            return 32 - self._b + 1
        return (w & -w).bit_length()

    @staticmethod
    def get_alpha_hll(m: int) -> float:
        if m == 16:
            return 0.673
        elif m == 32:
            return 0.697
        elif m == 64:
            return 0.709
        else:
            return 0.7213 / (1 + 1.079 / m)

In [4]:
b = 10
k = 1 << b
hashf = IntegerHash()

boundary = k * math.log(k)
test_cardinalities = [
    int(boundary * 0.1),
    int(boundary * 0.5),
    int(boundary),
    int(boundary * 2.0),
    int(boundary * 10.0)
]

print(f"Number of Registers: {k}")
print(f"Theoretical Boundry: ~ {boundary:.2f}")
print("-" * 80)
print(f"{'True N':<10} | {'HyperLog':<12} {'Err(%)':<8} | {'HyperLogLog':<12} {'Err(%)':<8} | {'Empty Registers':<10}")
print("-" * 80)

for n in test_cardinalities:
    sketch = HLLSketch(b=b, hashf=hashf)

    stream = list(range(n))
    random.shuffle(stream)
    for item in stream:
        sketch.add(item)

    est_hl = sketch.estimate_hyperlog()
    est_hll = sketch.estimate_hyperloglog()

    err_ll = abs(est_hl - n) / n * 100
    err_hll = abs(est_hll - n) / n * 100
    zeros = sketch.n_empty_registers()

    print(f"{n:<10} | {est_hl:<12.2f} {err_ll:<8.2f} | {est_hll:<12.2f} {err_hll:<8.2f} | {zeros:<10}")

print("-" * 80)

Number of Registers: 1024
Theoretical Boundry: ~ 7097.83
--------------------------------------------------------------------------------
True N     | HyperLog     Err(%)   | HyperLogLog  Err(%)   | Empty Registers
--------------------------------------------------------------------------------
709        | 858.32       21.06    | 1121.03      58.11    | 520       
3548       | 3449.59      2.77     | 3565.69      0.50     | 24        
7097       | 7059.80      0.52     | 6897.46      2.81     | 0         
14195      | 14283.01     0.62     | 13797.57     2.80     | 0         
70978      | 68914.54     2.91     | 71056.54     0.11     | 0         
--------------------------------------------------------------------------------
